# Fine-tune Qwen2.5-0.5B on MovieLens-1M movie metadata

Notebook version of [`train_qwen.py`](train_qwen.py). Every sample is a chat turn (system / user / assistant);
loss is computed on the assistant answer only.

| task | question | answer |
|---|---|---|
| `plot` | What is the movie "*title*" about? | Wikipedia plot (first `plot_words` words) |
| `genres` | Which genres does "*title*" belong to? | MovieLens genres |
| `details` | Who made "*title*" and who stars in it? | director + main cast |
| `identify` | Which movie is this? *plot snippet* | title (year) |
| `next_movie` *(optional)* | A user rated these movies … which next? | title (year) |

`next_movie` uses the same leave-last-out split as `interaction-analysis.ipynb`: the last 2 ratings of every user are
never trained on (2nd-to-last = eval, last = reserved for a later test).

**Run all** is safe: it builds the data, loads the model, checks label masking and runs one forward pass.
Training only starts when you set `RUN_TRAINING = True` in the config cell.

In [1]:
import json
import os
import random
import re
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path

# Apple GPU memory cache: by default it only frees cached blocks above ~1.4x the recommended working set
# (~18 GB on an 18 GB Mac), so it grows into swap and every step gets slower. Must be set before importing torch.
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.5")    # start recycling cached memory above this
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.9")   # hard cap: fail with OOM instead of swapping

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
METADATA_TASKS = ["plot", "genres", "details", "identify"]
ALL_TASKS = METADATA_TASKS + ["next_movie"]

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Config

In [ ]:
RUN_TRAINING = True   # <- set to True to actually train


@dataclass
class Config:
    model: str = "Qwen/Qwen2.5-0.5B"
    data_dir: Path = ROOT / "dataset" / "ml-1m"
    output_dir: Path = ROOT / "checkpoints" / "qwen2.5-0.5b-movielens"
    tasks: list = field(default_factory=lambda: list(METADATA_TASKS))   # add "next_movie" for recommendation
    # data
    plot_words: int = 150        # plot answers are cut to ~this many words
    snippet_words: int = 80      # plot snippet length for the identify task
    max_cast: int = 5
    history_len: int = 20        # next_movie: movies shown in the prompt
    min_history: int = 5         # next_movie: minimum history before a target
    next_per_user: int = 5       # next_movie: training targets sampled per user
    next_eval_users: int = 500   # next_movie: users in the eval set
    eval_frac: float = 0.02      # fraction of metadata samples held out for eval
    max_length: int = 512
    # training
    epochs: float = 2
    batch_size: int = 2          # on an 18 GB Mac: 2 peaks at ~7 GB GPU memory, 4 at ~11 GB (near the cap), ~4% faster
    grad_accum: int = 16         # effective batch = batch_size * grad_accum = 32
    lr: float = None             # default: 2e-5 full fine-tune, 2e-4 LoRA
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    eval_steps: int = 200
    save_steps: int = 400
    logging_steps: int = 10
    group_by_length: bool = True  # batch samples of similar length: much less padding
    gradient_checkpointing: bool = False   # only needed when memory is tight; costs ~30% speed
    # full fine-tune needs ~8 GB for weights + grads + AdamW and swaps on an 18 GB Mac (~7 h vs ~1 h for LoRA)
    lora: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    seed: int = 42

    def __post_init__(self):
        unknown = set(self.tasks) - set(ALL_TASKS)
        assert not unknown, f"unknown tasks {sorted(unknown)}; choose from {ALL_TASKS}"
        if self.lr is None:
            self.lr = 2e-4 if self.lora else 2e-5


cfg = Config()
set_seed(cfg.seed)
cfg

## 2. Build the samples

In [3]:
SYSTEM = "You are a movie expert who knows the MovieLens catalogue and recommends movies."
ARTICLES = r"The|A|An|La|Le|Les|L'|Il|El|Das|Der|Die"
ABBREVIATIONS = {"st", "dr", "mr", "mrs", "ms", "jr", "sr", "vs", "lt", "sgt", "capt", "col", "gen", "no"}
END = "<|im_end|>"

TEMPLATES = {
    "plot": ['What is the movie "{t}" about?', 'Summarize the plot of "{t}".', 'Tell me the story of the movie "{t}".'],
    "genres": ['Which genres does the movie "{t}" belong to?', 'What kind of movie is "{t}"?', 'List the genres of "{t}".'],
    "details": ['Who made the movie "{t}" and who stars in it?', 'Who directed "{t}" and who is in the cast?'],
    "identify": ["Which movie is this?\n\n{p}", "Name the movie with this plot:\n\n{p}"],
}


def display_title(ml_title):
    """'City of Lost Children, The (Cité des enfants perdus, La) (1995)' -> 'The City of Lost Children (1995)'"""
    m = re.match(r"^(.*?)\s*\((\d{4})\)\s*$", ml_title)
    name, year = (m.group(1), m.group(2)) if m else (ml_title, None)
    name = re.sub(r"\s*\([^()]*\)", "", name).strip() or name   # drop alternate titles
    name = re.sub(rf"^(.*), ({ARTICLES})$",
                  lambda a: a[2] + ("" if a[2].endswith("'") else " ") + a[1], name)
    return f"{name} ({year})" if year else name


def clean_plot(text):
    text = re.sub(r"\[\d+\]", "", str(text))   # citation markers
    return re.sub(r"\s+", " ", text).strip()


def truncate_words(text, n):
    """Cut to ~n words, ending on the last full sentence when that keeps at least half the text."""
    words = text.split()
    if len(words) <= n:
        return text
    cut = " ".join(words[:n])
    ends = [m.end() for m in re.finditer(r"[.!?][\"')]?(?=\s)", cut)
            if cut[:m.start()].split()[-1].lower().strip("\"'(") not in ABBREVIATIONS]
    return cut[:ends[-1]] if ends and ends[-1] > len(cut) * 0.5 else cut + " ..."


def mask_title(text, display):
    """Hide the movie's own name in an identify snippet so the answer isn't given away."""
    name = re.sub(r"\s*\(\d{4}\)$", "", display)
    for n in sorted({name, re.sub(r"^(The|A|An) ", "", name)}, key=len, reverse=True):
        if len(n) >= 4:   # skip very short names like "It" / "Big" that are ordinary words
            text = re.sub(rf"\b{re.escape(n)}\b", "___", text)
    return text


def is_known(value):
    return isinstance(value, str) and value.strip() and value.strip().lower() != "unknown"


def load_movies(data_dir):
    movies = pd.read_csv(data_dir / "movies_wiki.csv")
    movies["display"] = movies["Title"].map(display_title)
    # two different movies must never share a display title (it is the answer for identify / next_movie)
    dup = movies["display"].duplicated(keep=False)
    movies.loc[dup, "display"] = movies.loc[dup, "Title"]
    assert movies["display"].is_unique, movies.loc[movies["display"].duplicated(keep=False), "display"]
    return movies

In [4]:
def sample(task, prompt, answer):
    return {"task": task, "prompt": prompt, "answer": answer}


def metadata_samples(movies, tasks, cfg, rng):
    out = []
    for row in movies.to_dict("records"):
        t = row["display"]
        plot = clean_plot(row["Plot"]) if isinstance(row["Plot"], str) else None
        pick = lambda task: rng.choice(TEMPLATES[task])
        if "genres" in tasks:
            out.append(sample("genres", pick("genres").format(t=t), row["Genres"].replace("|", ", ")))
        if "plot" in tasks and plot:
            out.append(sample("plot", pick("plot").format(t=t), truncate_words(plot, cfg.plot_words)))
        if "identify" in tasks and plot:
            snippet = mask_title(truncate_words(plot, cfg.snippet_words), t)
            out.append(sample("identify", pick("identify").format(p=snippet), t))
        if "details" in tasks:
            parts = []
            if is_known(row["Director"]):
                parts.append(f"Directed by {row['Director'].strip()}.")
            if is_known(row["Cast"]):
                cast = [c.strip() for c in re.split(r",|\n", row["Cast"]) if c.strip()][: cfg.max_cast]
                parts.append(f"Starring {', '.join(cast)}.")
            if parts:
                out.append(sample("details", pick("details").format(t=t), " ".join(parts)))
    return out


def next_movie_prompt(history, display, genres):
    lines = [f"{k}. {display[m]} - {genres[m]}" for k, m in enumerate(history, 1)]
    return ("Here are the movies a user rated, oldest first:\n" + "\n".join(lines)
            + "\n\nWhich movie will they rate next? Answer with the title and year.")


def next_movie_samples(data_dir, movies, cfg, rng):
    """Leave-last-out split: train targets come from items[:-2]; eval target is items[-2]; items[-1] is never used."""
    ratings = pd.read_csv(data_dir / "ratings.dat", sep="::", engine="python",
                          names=["UserID", "MovieID", "Rating", "Timestamp"])
    seqs = ratings.sort_values(["UserID", "Timestamp"], kind="stable").groupby("UserID").MovieID.apply(list)
    display = dict(zip(movies.MovieID, movies.display))
    genres = dict(zip(movies.MovieID, movies.Genres.str.replace("|", ", ")))

    train, val = [], []
    for items in seqs:
        train_items = items[:-2]
        positions = list(range(cfg.min_history, len(train_items)))
        for t in rng.sample(positions, min(cfg.next_per_user, len(positions))):
            history = train_items[max(0, t - cfg.history_len):t]
            train.append(sample("next_movie", next_movie_prompt(history, display, genres), display[train_items[t]]))
        history = train_items[-cfg.history_len:]
        val.append(sample("next_movie", next_movie_prompt(history, display, genres), display[items[-2]]))
    rng.shuffle(val)
    return train, val[: cfg.next_eval_users]


def build_samples(cfg):
    rng = random.Random(cfg.seed)
    movies = load_movies(cfg.data_dir)
    meta = metadata_samples(movies, cfg.tasks, cfg, rng)
    rng.shuffle(meta)
    n_eval = int(len(meta) * cfg.eval_frac) if meta else 0
    train, evals = meta[n_eval:], {}
    if n_eval:
        evals["metadata"] = meta[:n_eval]
    if "next_movie" in cfg.tasks:
        nm_train, nm_val = next_movie_samples(cfg.data_dir, movies, cfg, rng)
        train += nm_train
        evals["next_movie"] = nm_val
    rng.shuffle(train)
    return train, evals

In [5]:
train_samples, eval_samples = build_samples(cfg)
print("train:", pd.Series([s["task"] for s in train_samples]).value_counts().to_dict())
print("eval: ", {k: len(v) for k, v in eval_samples.items()})
pd.DataFrame(train_samples).groupby("task").head(1).set_index("task")

train: {'genres': 3806, 'identify': 2760, 'plot': 2758, 'details': 2755}
eval:  {'metadata': 246}


,prompt,answer
task,,
details,"Who made the movie ""Adrenalin: Fear the Rush (...",Directed by Albert Pyun. Starring Christopher ...
plot,"Tell me the story of the movie ""A Star Is Born...",North Dakota farmgirl Esther Victoria Blodgett...
identify,Name the movie with this plot:\n\nSet on the b...,Butterfly Kiss (1995)
genres,"List the genres of ""Champagne (1928)"".",Comedy


## 3. Tokenize (loss on the answer only)

In [6]:
def prompt_text(tokenizer, prompt):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


class ChatDataset(Dataset):
    """Tokenized chat samples; labels are -100 on the system/user part so loss covers only the answer."""

    def __init__(self, samples, tokenizer, max_length):
        prompts = tokenizer([prompt_text(tokenizer, s["prompt"]) for s in samples], add_special_tokens=False).input_ids
        answers = tokenizer([s["answer"] + END for s in samples], add_special_tokens=False).input_ids
        self.items, self.dropped = [], 0
        for s, p, a in zip(samples, prompts, answers):
            if len(p) >= max_length - 8:   # prompt alone fills the context: nothing left to learn from
                self.dropped += 1
                continue
            ids = (p + a)[:max_length]
            labels = ([-100] * len(p) + a)[:max_length]
            self.items.append({"input_ids": ids, "labels": labels, "task": s["task"]})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        item = self.items[i]
        return {"input_ids": item["input_ids"], "labels": item["labels"]}


class PadCollator:
    """Pads to a multiple of `multiple` so the GPU memory cache sees a few batch shapes instead of hundreds."""

    def __init__(self, pad_id, multiple=32):
        self.pad_id = pad_id
        self.multiple = multiple

    def __call__(self, batch):
        n = max(len(b["input_ids"]) for b in batch)
        n = -(-n // self.multiple) * self.multiple
        pad = lambda seq, value: seq + [value] * (n - len(seq))
        return {
            "input_ids": torch.tensor([pad(b["input_ids"], self.pad_id) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
        }


def describe(name, ds):
    lengths = sorted(len(x["input_ids"]) for x in ds.items)
    tasks = pd.Series([x["task"] for x in ds.items]).value_counts().to_dict()
    print(f"{name}: {len(ds)} samples (dropped {ds.dropped}), tokens median {lengths[len(lengths) // 2]}, "
          f"p95 {lengths[int(len(lengths) * .95)]}, max {lengths[-1]}, tasks {tasks}")

In [7]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model)
train_ds = ChatDataset(train_samples, tokenizer, cfg.max_length)
eval_ds = {k: ChatDataset(v, tokenizer, cfg.max_length) for k, v in eval_samples.items()}
describe("train", train_ds)
for k, v in eval_ds.items():
    describe(f"eval/{k}", v)

train: 12079 samples (dropped 0), tokens median 77, p95 238, max 290, tasks {'genres': 3806, 'identify': 2760, 'plot': 2758, 'details': 2755}
eval/metadata: 246 samples (dropped 0), tokens median 79, p95 239, max 254, tasks {'genres': 77, 'plot': 58, 'identify': 56, 'details': 55}


In [8]:
# check the masking: the full chat vs. the part that is actually in the loss, one example per task
first = {}
for i, item in enumerate(train_ds.items):
    first.setdefault(item["task"], i)
for task, i in first.items():
    item = train_ds.items[i]
    answer_ids = [t for t in item["labels"] if t != -100]
    print(f"--- {task} ({len(item['input_ids'])} tokens, {len(answer_ids)} in loss)")
    print(tokenizer.decode(item["input_ids"]))
    print(f"[LOSS ON] {tokenizer.decode(answer_ids)!r}\n")

--- details (69 tokens, 17 in loss)
<|im_start|>system
You are a movie expert who knows the MovieLens catalogue and recommends movies.<|im_end|>
<|im_start|>user
Who made the movie "Adrenalin: Fear the Rush (1996)" and who stars in it?<|im_end|>
<|im_start|>assistant
Directed by Albert Pyun. Starring Christopher Lambert, Natasha Henstridge.<|im_end|>
[LOSS ON] 'Directed by Albert Pyun. Starring Christopher Lambert, Natasha Henstridge.<|im_end|>'

--- plot (232 tokens, 186 in loss)
<|im_start|>system
You are a movie expert who knows the MovieLens catalogue and recommends movies.<|im_end|>
<|im_start|>user
Tell me the story of the movie "A Star Is Born (1937)".<|im_end|>
<|im_start|>assistant
North Dakota farmgirl Esther Victoria Blodgett (Janet Gaynor) yearns to become a Hollywood actress. Although her aunt and father discourage such thoughts, Esther's grandmother (May Robson) gives Esther her savings to follow her dream. Esther goes to Hollywood and tries to land a job as an extra, but

## 4. Load the model

In [9]:
def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_model(cfg, device):
    use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(cfg.model, dtype=torch.bfloat16 if use_bf16 else torch.float32)
    model.config.use_cache = False
    if cfg.lora:
        from peft import LoraConfig, get_peft_model
        model = get_peft_model(model, LoraConfig(
            r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
        if cfg.gradient_checkpointing:
            model.enable_input_require_grads()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"model {cfg.model}: {total / 1e6:.0f}M params, {trainable / 1e6:.1f}M trainable, "
          f"dtype {next(model.parameters()).dtype}, device {device}")
    return model, use_bf16


@torch.no_grad()
def show_generations(model, tokenizer, samples, device, max_new_tokens=80):
    model.eval()
    for s in samples:
        enc = tokenizer(prompt_text(tokenizer, s["prompt"]), return_tensors="pt", add_special_tokens=False).to(device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=tokenizer.convert_tokens_to_ids(END), pad_token_id=tokenizer.pad_token_id)
        pred = tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"--- [{s['task']}] {s['prompt'][:120]!r}\n    expected:  {s['answer'][:120]!r}\n    generated: {pred[:120]!r}")


device = pick_device()
model, use_bf16 = load_model(cfg, device)
collator = PadCollator(tokenizer.pad_token_id)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 754.19it/s]

model Qwen/Qwen2.5-0.5B: 503M params, 8.8M trainable, dtype torch.float32, device mps


## 5. Sanity check before training

In [10]:
model.to(device)
batch = {k: v.to(device) for k, v in collator([train_ds[i] for i in range(cfg.batch_size)]).items()}
t0 = time.time()
with torch.no_grad():
    loss = model(**batch).loss
print(f"forward pass on one batch {tuple(batch['input_ids'].shape)}: loss {loss.item():.3f} ({time.time() - t0:.2f}s)")

forward pass on one batch (4, 256): loss 3.257 (0.91s)


In [11]:
# what the base model answers before fine-tuning
picks = [next(s for s in train_samples if s["task"] == t) for t in first]
picks += [v[0] for v in eval_samples.values()]
show_generations(model, tokenizer, picks, device, max_new_tokens=40)

--- [details] 'Who made the movie "Adrenalin: Fear the Rush (1996)" and who stars in it?'
    expected:  'Directed by Albert Pyun. Starring Christopher Lambert, Natasha Henstridge.'
    generated: 'The movie "Adrenalin: Fear the Rush (1996)" was made by the company "Columbia Pictures". The movie stars the actors "Rob'
--- [plot] 'Tell me the story of the movie "A Star Is Born (1937)".'
    expected:  'North Dakota farmgirl Esther Victoria Blodgett (Janet Gaynor) yearns to become a Hollywood actress. Although her aunt an'
    generated: 'A Star Is Born (1937) is a 1937 American romantic drama film directed by John Ford and starring Humphrey Bogart and Ingr'
--- [identify] 'Name the movie with this plot:\n\nSet on the bleak motorways of Lancashire, ___ tells the story of Eunice (Plummer), a bis'
    expected:  'Butterfly Kiss (1995)'
    generated: 'The movie is about a serial killer named Eunice who is a bisexual and a serial killer. The story revolves around Miriam,'
--- [genres] 'List

## 6. Trainer

In [ ]:
class AnswerOnlyTrainer(Trainer):
    """Same loss as the default Trainer, but the 151k-vocab output layer only runs on answer tokens.

    The default computes fp32 logits for every token (~700 MB per copy for a long batch) although only the answer
    is in the loss; on Apple GPUs that transient memory piles up in the allocator cache until it hits the limit.
    """

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        lm = model.get_base_model() if hasattr(model, "get_base_model") else model
        targets = inputs["labels"][:, 1:]
        keep = targets != -100
        with self.accelerator.autocast():   # calling the inner modules skips accelerate's fp16/bf16 forward wrapper
            hidden = lm.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
            logits = lm.lm_head(hidden[:, :-1][keep]).float()
        loss = torch.nn.functional.cross_entropy(logits, targets[keep], reduction="sum")
        # num_items_in_batch = answer tokens over all accumulated micro-batches (how the default loss normalises)
        loss = loss / (num_items_in_batch if num_items_in_batch is not None else keep.sum())
        return (loss, {"logits": logits}) if return_outputs else loss


steps_per_epoch = -(-len(train_ds) // (cfg.batch_size * cfg.grad_accum))
print(f"{steps_per_epoch} optimizer steps/epoch x {cfg.epochs} epochs "
      f"(effective batch {cfg.batch_size * cfg.grad_accum}, lr {cfg.lr})")

training_args = TrainingArguments(
    output_dir=str(cfg.output_dir),
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    lr_scheduler_type="cosine",
    warmup_steps=cfg.warmup_ratio,   # transformers v5: a float in [0, 1) is a ratio of total steps
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    eval_strategy="steps" if eval_ds else "no",
    eval_steps=cfg.eval_steps,
    save_strategy="steps",
    save_steps=cfg.save_steps,
    save_total_limit=2,
    bf16=use_bf16,
    gradient_checkpointing=cfg.gradient_checkpointing,
    train_sampling_strategy="group_by_length" if cfg.group_by_length else "random",
    torch_empty_cache_steps=10,      # release cached GPU memory so it doesn't creep into swap
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
    seed=cfg.seed,
)
trainer = AnswerOnlyTrainer(model=model, args=training_args, train_dataset=train_ds,
                            eval_dataset=eval_ds or None, data_collator=collator, processing_class=tokenizer)
trainer.create_optimizer()   # catches optimizer / device issues without taking a step
print(f"optimizer: {type(trainer.optimizer).__name__}")

# the answer-only loss must match the model's own loss (eval mode: no LoRA dropout)
model.eval()
with torch.no_grad():
    check = {k: v.to(device) for k, v in collator([train_ds[i] for i in range(cfg.batch_size)]).items()}
    ours, ref = trainer.compute_loss(model, check).item(), model(**check).loss.item()
print(f"loss check: answer-only {ours:.4f} vs default {ref:.4f}")
assert abs(ours - ref) < 1e-3

## 7. Train

Only runs when `RUN_TRAINING = True` (config cell). Checkpoints go to `cfg.output_dir` every `save_steps`;
the final model is saved to `cfg.output_dir / "final"`. To resume after an interruption use
`trainer.train(resume_from_checkpoint=True)`.

In [ ]:
final_dir = cfg.output_dir / "final"
if RUN_TRAINING:
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    with open(cfg.output_dir / "run_config.json", "w") as f:
        json.dump({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, f, indent=2)
    trainer.train()
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f"saved to {final_dir}")
else:
    print("RUN_TRAINING is False - skipping training")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 10.75 GiB, other allocations: 967.72 MiB, max allowed: 11.99 GiB). Tried to allocate 672.00 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

: 

## 8. After training

In [ ]:
if RUN_TRAINING:
    if eval_ds:
        print(trainer.evaluate())
    log = pd.DataFrame(trainer.state.log_history)
    display(log.dropna(subset=["loss"])[["step", "loss"]].tail())
    show_generations(model, tokenizer, picks, device)

### Reload the fine-tuned model later

```python
tokenizer = AutoTokenizer.from_pretrained(final_dir)
model = AutoModelForCausalLM.from_pretrained(final_dir).to(device)   # with LoRA: peft.AutoPeftModelForCausalLM
show_generations(model, tokenizer, picks, device)
```